# Step 5: Scale & Feature Engineering (Co Giãn & Tạo Đặc Trưng)
### Theo Quy Trình 8 Bước Từ Slide Bài Giảng Chapter 4 (NEU FIT)

---

### 1. Phân Tích Chuyên Sâu Cách Làm:
Slide 29 & 30 giải thích lý do cần Scale:
- **Vì sao phải scale?** Khoảng cách giữa các biến có đơn vị khác nhau (ví dụ: diện tích $100	ext{m}^2$ vs số lượng giao dịch $15$) sẽ khiến các mô hình tuyến tính (Ridge, Logistic) hoặc khoảng cách (KNN, SVM) bị chi phối bởi biến có thang đo lớn hơn.
- **Rule 2 (Slide 61):** *Fit on train only! Imputer and scaler: fit on train, transform test; a pipeline does it for you.*
- **Biến đổi hàm mục tiêu:** Doanh thu có phân phối lệch phải (skewed right). Áp dụng logarit tự nhiên $\log(1 + y)$ đưa về phân phối chuẩn, giúp RMSE trên không gian log khớp chính xác với hàm mục tiêu **RMSLE** của cuộc thi!
- **Feature Engineering đặc thù cho Quý 1/2026:**
  - Khuyến mãi thực tế: `is_promo_global`, `promo_weekend`, `promo_{store_type}`.
  - Phân kỳ Tết 10 giai đoạn (Tết Bính Ngọ 17/02/2026).
  - Tương tác Lễ Tình Nhân Valentine 14/02 và Quốc tế Phụ Nữ 08/03.
  - Phân tách phi tuyến thời tiết: mưa nhẹ ($0-15	ext{mm}$) vs mưa ngập ($>15	ext{mm}$).

### 2. Dùng Những Gì Để Làm?
- **Thư viện:** `sklearn.preprocessing` (`StandardScaler`, `MinMaxScaler`), `numpy` (`log1p`).

In [1]:
import os, sys
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

for p in [os.path.abspath(".."), os.path.abspath("."), os.path.abspath("../..")]:
    if os.path.exists(os.path.join(p, "data")):
        DATA_DIR = os.path.join(p, "data")
    if p not in sys.path and os.path.exists(os.path.join(p, "src")):
        sys.path.insert(0, p)

from src.cleaning import clean_train, clean_stores
from src.features import enrich_data, build_shared_matrix

# Đọc và làm sạch
stores_c = clean_stores(pd.read_csv(os.path.join(DATA_DIR, "stores.csv")))
train_c, _ = clean_train(pd.read_csv(os.path.join(DATA_DIR, "train.csv")), stores_c)

# 1. Biến đổi log1p trên biến mục tiêu
train_c["log_revenue"] = np.log1p(train_c["revenue"])

# 2. Chuẩn hóa đặc trưng số bằng StandardScaler (Fit on train only)
num_cols = ["seats", "area_m2"]
train_merged = train_c.merge(stores_c, on="store_id", how="left")

scaler = StandardScaler()
scaled_features = scaler.fit_transform(train_merged[num_cols])
print("Mean đã học trên train :", scaler.mean_.round(2))
print("Std đã học trên train  :", np.sqrt(scaler.var_).round(2))

display(pd.DataFrame(scaled_features, columns=[f"{c}_scaled" for c in num_cols]).head(4))

print(">>> HOÀN THÀNH STEP 5: ĐÃ CHUẨN HÓA LOG VÀ SCALE ĐẶC TRƯNG CHUẨN XÁC!")

Mean đã học trên train : [ 50.83 134.07]
Std đã học trên train  : [17.88 43.  ]


,seats_scaled,area_m2_scaled
0,-1.276687,-0.908670
1,-0.717366,-0.327297
2,-0.549570,-0.769140
3,1.352122,1.323800


>>> HOÀN THÀNH STEP 5: ĐÃ CHUẨN HÓA LOG VÀ SCALE ĐẶC TRƯNG CHUẨN XÁC!
